In [ ]:
# !pip install pylabwons
!pip install pylabwons --upgrade --no-deps

In [ ]:
from datetime import datetime
import pandas as pd
import pandas_datareader.data as web
import plotly.io as pio
import plotly.graph_objects as go
import pylabwons as lw
import yfinance as yf
pio.renderers.default = "vscode"

clock = datetime.now()

# ASSET




| 구분 | **기초 자산** | **가중 방식** | **운용 방식** | **주요 종목** | **주요 특징** |
| :--- | :--- | :--- | :--- | :--- | :--- |
| `XLE` (대형주 중심) | S&P 500 에너지 섹터 기업 | **시가총액 가중** (대형주 비중⬆️) | 1배수 (현물 ETF) | 엑손모빌, 쉐브론 등 대기업 | 안정적인 대형주 및 배당 중심 |
| `XOP` (탐사/생산 중심) | S&P Oil & Gas E&P 지수 | **동일 가중** (모든 종목 균등 비중) | 1배수 (현물 ETF) | 중소형 탐사·생산(E&P) 기업 | 유가 변동에 민감한 중소형주 포함 |
| `ERX` (🧲 XLE) | Energy Select Sector 지수 | **시가총액 가중** (대형주 비중⬆️) | **일일 수익률 정방향 2배 (ETF)** | 엑손모빌, 쉐브론 등 대기업 | 대형 에너지주 상승 시 레버리지 수익 |
| `ERY` (🧲 XLE) | Energy Select Sector 지수 | **시가총액 가중** (대형주 비중⬆️) | **일일 수익률 역방향 -2배 (ETF)** | 엑손모빌, 쉐브론 등 대기업 | 대형 에너지주 하락 시 인버스 수익 |
| `GUSH` (🧲 XOP) | S&P Oil & Gas E&P 지수 | **동일 가중** (모든 종목 균등 비중) | **일일 수익률 정방향 2배 (ETF)** | 중소형 탐사·생산(E&P) 기업 | 유가 상승기 중소형주 강세 시 높은 수익 |
| `DRIP` (🧲 XOP) | S&P Oil & Gas E&P 지수 | **동일 가중** (모든 종목 균등 비중) | **일일 수익률 역방향 -2배 (ETF)** | 중소형 탐사·생산(E&P) 기업 | 유가 하락기 중소형주 약세 시 높은 수익 |
| `WTIU` | Solactive MicroSectors US Big Oil Index | **동일 가중** (상위 12개 대형주 중심) | **일일 수익률 3배 추종 (ETN)** | 엑손모빌, 쉐브론 등 우량주 12개 | 단기 방향성 매매용 (변동성↗️) |
| `USO` (원유 선물) | WTI(서부 텍사스산 원유) 선물 | **해당 없음** (선물 계약 중심) | 1배수 (선물 ETF) | WTI 원유 선물 | 원유 가격(선물) 직접 추종 (롤오버 비용 발생) |



In [ ]:
TICKERS = lw.DataDict(**{
    ticker: yf.Ticker(ticker) for ticker in [
        'XLE', 
        'XOP', 
        'ERX', 
        'ERY', 
        'GUSH', 
        'DRIP', 
        'WTIU', 
        'USO'
]})

objs = {}
for ticker in TICKERS.values():
    ohlcv = ticker.history(period='10y', interval='1d')[['Open', 'High', 'Low', 'Close', 'Volume']]
    ohlcv.columns = ['open', 'high', 'low', 'close', 'volume']
    objs[ticker.ticker] = ohlcv
    

asset = pd.concat(objs, axis=1)
asset = asset.tz_localize(None) if asset.index.tz is not None else asset
# asset

In [ ]:
mar = lw.MultiAssetRelation(asset.xs('close', axis=1, level=1))
# mar.cummulative_return
# mar.drawdown
fig = mar.plotly_multi('cumulative_return')
fig.update_layout(height=600)

# MACRO

## INDICATORS LIST

| 번호 | 지표명 (Indicator) | 데이터 출처 (Source) | 업데이트 주기 | 분석 내재 의미 및 활용법 (Core Interpretation) |
| :--- | :--- | :--- | :--- | :--- |
| **1** | **상업용 원유 재고**<br>(Commercial Crude Inventories) | 미국 에너지정보청<br>([EIA](https://eia.gov)) | 매주 수요일<br>(주간) | - 미국 내 단기 원유 공급 과잉 및 부족을 가늠하는 척도<br>- 계절성(5년 평균치) 대비 급감 시 유가 상방 압력 |
| **2** | **SPR 원유 재고**<br>(Strategic Petroleum Reserve) | 미국 에너지정보청<br>([EIA](https://eia.gov)) | 매주 수요일<br>(주간) | - 정부의 인위적 시장 개입(방출/재비축)을 나타내는 지표<br>- 상업용 재고와 합산한 '총 재고량'으로 공급 실태 파악 필수 |
| **3** | **Baker Hughes Rig Count**<br>(시추기 수) | 베이커 휴즈 공시<br>([Baker Hughes](https://bakerhughes.com)) | 매주 금요일<br>(주간) | - 미국 셰일오일 공급의 최선행 지표 (유가 변동에 3~6개월 후행)<br>- 유가 상승에도 Rig Count 증가가 둔화되면 공급 부족 지속 시그널 |
| **4** | **`CL=F`, `BZ=F`, `NG=F`**<br>(WTI, Brent, 천연가스 선물) | 야후 파이낸스<br>([Yahoo Finance](https://yahoo.com)) | 실시간<br>(일간/분간) | - 에너지 원자재 시장의 벤치마크 가격 지표<br>- 근월물과 원월물 간의 가격 차이(백워데이션/콘탱고)로 시장 심리 진단 |
| **5** | **생산자 물가지수: Drilling PPI**<br>(PCU213111213111) | 세인트루이스 연은<br>([FRED](https://stlouisfed.org)) | 매월 중순<br>(월간) | - 유전 시추 및 개발에 드는 물리적 비용 지수<br>- 유가가 높아도 이 지표가 급등하면 기업의 실질 마진(FCF) 훼손 의미 |
| **6** | **`DXY` 달러 인덱스**<br>(U.S. Dollar Index) | 야후 파이낸스<br>([Yahoo Finance](https://yahoo.comquote/DX-Y.NYB/)) | 실시간<br>(일간/분간) | - 원유의 결제 통화인 달러화의 절대 가치<br>- 유가와 강한 역상관관계 (`DXY` 하락 = 유가 상승 지지 환경) |
| **7** | **북미 정제시설 가동률**<br>(Refinery Utilization Rate) | 미국 에너지정보청<br>([EIA](https://eia.gov)) | 매주 수요일<br>(주간) | - 정유사들이 원유를 사들이는 '실제 수요'의 강도를 증명<br>- 90% 이상 유지 시 원유 소화력 양호, 하락 시 원유 수요 둔화 경고 |
| **8** | **Crack Spread (3-2-1)**<br>(정제마진 직접 계산) | Yahoo Finance 데이터 가공<br>(`CL=F`, `RB=F`, `HO=F` 활용) | 실시간<br>(일간) | - 정유사(XLE 등 주요 기업)의 핵심 수익성 직결 지표<br>- 유가 고점 대비 마진 선행 하락 시 에너지 주식 탈출(Exit) 신호 |

---

## CHECK-LIST & GUIDELINES

본 가이드라인은 대형주 중심(XLE) 및 생산/탐사 중심(XOP) ETF 투자 시, 레버리지/인버스 진입 및 청산 타이밍을 계량화하기 위한 매크로 지표 조합 분석 템플릿입니다.

---

<h3>📈 LONG & LEVERAGE</h3>

아래 지표 조합 중 **4개 이상이 충족**되거나, **핵심 조합(수요-정제-가격)이 일치**할 경우 1배수 진입 또는 단기 레버리지(2x, 3x) 전략 검토


| 대분류 | ☑️ 체크 항목 | 시장 판단 및 투자 시그널 |
| :--- | :--- | :--- |
| ⚠️**수요 및 정제 마진** | ⬜ Crack Spread 상승 + 정제시설 가동률(Utilization Rate) 상승 | - 최종 제품(휘발유, 등유 등) 수요 강력<br>- 정제 마진 개선 국면<br>- **XLE, XOP 강력한 매수 신호** |
| ⚠️**수요 및 정제 마진** | ⬜ 상업용 원유 재고 감소 (EIA)<br>⬜ SPR 재고 증가 동시 발생 (EIA) | - 5년 평균 대비 재고 하락 추세 또는 예상치보다 큰 감소 폭<br>- **단기 유가(CL=F, BZ=F) 상승 압력 작용** |
| **수요** | ⬜ BZ=F(브렌트)와 CL=F(WTI) 스프레드(Spread) 확대 | - 글로벌 기준물(브렌트)이 미국 유가보다 강세<br>- 미국 외 글로벌 실물 수요 탄탄 (수출 마진 확대로 **미국 에너지 기업 호재**) |
| **수요** | ⬜ Crack Spread 상승 속 원유 선물(CL=F, BZ=F) 백워데이션(Backwardation) 심화 | - 근월물 가격이 원월물보다 비싼 현상 심화 (실물 원유 극심한 부족)<br>- **숏 스퀴즈 유발 가능성 상승으로 레버리지 진입 최적기** |
| **중장기 공급 및 생산 기조** | ⬜ Baker Hughes Rig Count 정체/감소 + 생산자 물가지수(PPI: Drilling) 상승 | - 시추 비용(PPI) 상승에도 시추기 수(Rig Count) 정체<br>- 공급 과잉 우려 해소 및 고유가 유지 환경 조성 (**XOP 호재**) |
| **중장기 공급 및 생산 기조** | ⬜ SPR(전략비축유) 재고 고정 또는 매입 전환 (EIA) | - 미 정부의 SPR 방출 종료 및 재충전(매입) 단계 진입<br>- **유가 하방 지지선 형성** |
| **매크로 및 금융 자산 동향** | ⬜ 달러 인덱스(DXY) 하락 추세 전환 | - 원유의 달러 결제 특성 반영<br>- **CL=F, BZ=F 가격의 명목적 상승 유발** |
| **매크로 및 금융 자산 동향** | ⬜ 천연가스 선물(NG=F) 계절적 반등 또는 CL=F 동반 상승 | - **에너지 섹터 전반의 투자 심리(Sentiment) 개선** |

---

<h3>📉 SHORT / INVERSE / CASH OUT: EXIT</h3>

아래 위험 신호 중 **3개 이상이 동시 발생**하거나, **정제 마진 붕괴가 확인**되면 포지션을 축소(수익 실현)하거나 단기 인버스 상품 진입을 검토합니다.


| 대분류 | ☑️ 체크 항목 | 시장 판단 및 매도/인버스 시그널 |
| :--- | :--- | :--- |
| ⚠️**수요 둔화** | ⬜ Crack Spread 급락 (정제시설 가동률 고점 통과) | - 유가가 높더라도 정제 마진 붕괴 시 정유사(XLE) 실적 악화<br>- **원유 수요 감소의 전조 증상** |
| ⚠️**수요 둔화** | ⬜ 상업용 원유 재고(EIA)의 지속적인 누적 | - 공급이 수요를 초과하기 시작했다는 확실한 증거<br>- **지표 발표 시마다 유가 하방 압력 강화** |
| **수요 둔화** | ⬜ 정제시설 가동률 하락 속 상업용 원유 재고 증가 | - 가동률을 낮춤에도 재고가 쌓이는 원유 수요 둔화 상태<br>- **유가 폭락 신호탄으로 즉시 현금화 또는 인버스 준비** |
| **공급 과잉 및 생산 단가 하락** | ⬜ Baker Hughes Rig Count 급증 + 생산자 물가지수(PPI: Drilling) 안정화 | - 시추 비용 하락 및 공급(Rig) 증가 국면<br>- **셰일 기업 중심인 XOP의 급락 유발 가능** |
| **공급 과잉 및 생산 단가 하락** | ⬜ SPR 원유 재고의 대규모 방출 지속 | - 정부의 강제적인 시장 공급 물량 확대 구간<br>- **유가 상방 유의미하게 제한** |
| **공급 과잉 및 생산 단가 하락** | ⬜ CL=F, BZ=F 가격 상승 속 Crack Spread 나홀로 급락 (다이버전스) | - 제품 소비자가 고유가를 버티지 못해 마진이 깨지는 현상<br>- 가짜 상승일 확률이 높으며 **정유주(XLE) 급락 임박 시사** |
| **공급 과잉 및 생산 단가 하락** | ⬜ Baker Hughes Rig Count 증가 전환 + 원유 선물 콘탱고(Contango) 진입 | - 시추기 수 증가 및 콘탱고(원월물>근월물) 변동 발생<br>- 공급 과잉 우려가 시장을 지배하는 **확실한 숏(Short) 신호** |
| **역풍(Headwind) 및 자산 붕괴** | ⬜ 달러 인덱스(DXY) 강한 반등 또는 고점 돌파 | - 글로벌 유동성 위축 발생<br>- **원유 가격의 명목적 하락 압력으로 작용** |
| **역풍(Headwind) 및 자산 붕괴**| ⬜ CL=F, BZ=F 선물 가격의 데드크로스 (기술적 추세 이탈) | - 매크로 지표 악화와 함께 주요 이평선(50일, 200일) 이탈<br>- **레버리지 포지션 즉시 청산 신호** |
| **매크로 및 생산 단가 하락** | ⬜ DXY 급등과 동시에 Drilling PPI 하락 전환 | - 마진 한계선인 생산 비용 하락 및 달러 강세 동시 진행<br>- 섹터 마진 스프레드가 압박받아 **주가 폭락 유발** |

---

💡 레버리지 / 인버스 특화 속성 가이드 (XLE vs XOP)

* **XLE**: **`Crack Spread`**와 **`DXY`**의 영향력이 절대적입니다. 정제 마진이 유지되는 한 유가 변동성이 커도 배당과 안정성으로 버티는 경향이 있습니다. 특히 `DXY 급등` + `Crack Spread 낙폭 과대` 시에는 대형주 실적 훼손이 빠르므로 **XLE 인버스**가 안전합니다.
* **XOP**: **`CL=F(WTI)`**, **`Baker Hughes Rig Count`**, **`Drilling PPI`**에 매우 민감합니다. 공급 통제 속에서 유가가 급등하거나 `백워데이션 심화` + `Crack Spread 상승`이 동반될 때 **XOP 레버리지**의 효율이 극대화됩니다. 반대로 공급 과잉 신호(`가동률 하락` + `상업용 재고 증가`) 시에는 **XOP 인버스**의 변동성이 XLE보다 훨씬 큽니다.


In [ ]:
def get_futures_ticker(symbol:str, months_ahead:int):
    dt = datetime.now()
    kw = {1:'F', 2:'G', 3:'H', 4:'J', 5:'K', 6:'M', 7:'N', 8:'Q', 9:'U', 10:'V', 11:'X', 12:'Z'}
    total_months = dt.month - 1 + months_ahead
    mm, yy = (total_months % 12) + 1, dt.year + (total_months // 12)    
    return f'{symbol}{kw[mm]}{str(yy)[-2:]}.NYM'

future = get_futures_ticker

crack_spread = lw.fetch.crack_spread(year=10)
# crack_spread

eia = lw.fetch.eia() # ['commercial_stocks', 'SPR_stocks', 'utilization_rate']
# eia

oil_price = yf.Tickers([
    'BZ=F', 
    'CL=F', 
    'NG=F', 
    future('CL', 1),
    future('CL', 3),
    # future('BZ', 1),
    future('BZ', 3),
]).history(period='10y', interval='1d')['Close']
# oil_price

rig = lw.fetch.baker_hughes_rig_count() # fetch time: ~20s
# rig

ppi = web.DataReader('PCU213111213111', 'fred', datetime(clock.year - 10, clock.month, clock.day), clock)
# ppi

dxy = yf.Ticker('DX-Y.NYB').history(period='10y', interval='1d')['Close']
# dxy

### ☑️ 수요와 재고

*   **상업용 원유 재고 (Commercial Crude Inventories)**(단위: Thousand Barrels)<br>
민간 정유사 및 저장 시설이 보유한 재고. 단기 원유 시장의 수요 & 공급 상태를 가장 잘 나타냄
*   **전략비축유 재고 (SPR, Strategic Petroleum Reserve)**(단위: Thousand Barrels)<br>
미국 정부의 전략비축유(텍사스와 루이지애나 지하 암염동굴 보관). 정부의 정책(인위)적 개입 수단

---

| 구분 (행) | 재고 증가 (Increase) | 재고 감소 (Decrease) |
| :--- | :--- | :--- |
| **상업용 원유 재고**<br>(Commercial Inventories) | **📉 Short (매도) 신호**<br>· 공급 과잉 또는 수요 둔화 시그널<br>· 유가 하락 압력 ➔ 에너지 섹터 하락 | **📈 Long (매수) 신호**<br>· 수요 강세 또는 공급 부족 시그널<br>· 유가 상승 압력 ➔ 에너지 섹터 상승 |
| **SPR 재고**<br>(Strategic Petroleum Reserve) | **📈 Long (매수) 신호**<br>· 정부의 비축유 재구입(매입 수요 발생)<br>· 유가 하방 지지선 역할 ➔ 에너지 섹터 상승 | **📉 Short (매도) 신호**<br>· 정부의 비축유 방출(시장 공급 증가)<br>· 유가 고점 통제 목적 ➔ 에너지 섹터 하락 |


In [ ]:
comm, spr = eia['commercial_stocks'], eia['SPR_stocks']
comm.name, spr.name = 'Commercial', 'SPR'
factor1 = lw.DualRelation(comm, spr)
fig = factor1.show(height=600)
fig.add_hline(
    secondary_y=False,
    y=comm[comm.index >= datetime(clock.year - 5, clock.month, clock.day)].mean(),
    line_width=0.8,
    line_dash='dot'
)

### ☑️ 가격 스프레드와 정제 마진

*   **크랙 스프레드 (Crack Spread)**(단위: $/bbl)<br>
원유 가격과 정제된 석유 제품(휘발유, 등유 등) 가격의 차이. 정유사의 수익성을 나타내는 핵심 지표
*   **브렌트유-WTI유 스프레드 (Brent-WTI Spread)**(단위: $/bbl)<br>
국제 기준 원유인 브렌트유와 미국 서부텍사스산원유(WTI)의 가격 차이. 글로벌 공급 상태와 미국 원유 수출 경쟁력을 나타냄

---


| 구분 (행) | 스프레드 증가 / 확대 (Increase) | 스프레드 감소 / 축소 (Decrease) |
| :--- | :--- | :--- |
| **크랙 스프레드**<br>(Crack Spread) | **📈 Long (매수) 신호**<br>· 최종 석유 제품 수요 강세 시그널<br>· 정유사 마진/수익성 급증 ➔ 에너지 섹터 상승 | **📉 Short (매도) 신호**<br>· 석유 제품 공급 과잉 또는 수요 둔화 시그널<br>· 정유사 마진 축소 및 실적 악화 ➔ 에너지 섹터 하락 |
| **브렌트유-WTI유 스프레드**<br>(Brent-WTI Spread) | **📈 Long (매수) 신호**<br>· 미국 외 지역 공급 불안 또는 미 원유 수출 수요 증가<br>· 미국 정유사/수출기업 반사이익 ➔ 에너지 섹터 상승 | **📉 Short (매도) 신호**<br>· 글로벌 공급 과잉 또는 미국 내 공급 병목 발생<br>· 미국 원유의 가격 경쟁력 약화 ➔ 에너지 섹터 하락 |


In [ ]:
spread = oil_price['BZ=F'] - oil_price['CL=F']
spread.name = 'Brent-WTI Spread'
factor2 = lw.DualRelation(crack_spread, spread)
factor2.show()

### ☑️ 생산 활동과 가동률

*   **베이커 휴즈 시추기 수 (Baker Hughes Rig Count)**(단위: 기, Rigs)<br>
미국 내에서 가동 중인 원유 및 가스 시추기 수. 향후 원유 생산량의 증감을 예측하는 선행 지표
*   **EIA 정제 시설 가동률 (EIA Refinery Utilization Rate)**(단위: %)<br>
미국 정유 공장들이 보유한 전체 정제 설비 중 실제 가동 중인 비율. 원유의 단기 수요 강도를 나타냄

---


| 구분 (행) | 지표 증가 / 상승 (Increase) | 지표 감소 / 하락 (Decrease) |
| :--- | :--- | :--- |
| **베이커 휴즈 시추기 수**<br>(Baker Hughes Rig Count) | **📉 Short (매도) 신호**<br>· 향후 원유 공급량 증가(생산 과잉) 시그널<br>· 중장기 유가 하락 압력 ➔ 에너지 섹터 하락 | **📈 Long (매수) 신호**<br>· 향후 원유 공급량 감소(생산 위축) 시그널<br>· 중장기 유가 상승 압력 ➔ 에너지 섹터 상승 |
| **EIA 정제 시설 가동률**<br>(EIA Utilization Rate) | **📈 Long (매수) 신호**<br>· 정유사의 원유 매입 및 처리 수요 강세 시그널<br>· 단기 유가 상승 유발 ➔ 에너지 섹터 상승 | **📉 Short (매도) 신호**<br>· 정유사 유지보수(계절적 요인) 또는 수요 둔화<br>· 단기 원유 재고 쌓임 유발 ➔ 에너지 섹터 하락 |


In [ ]:
rig_count, utilization_rate = rig['Total'], eia['utilization_rate']
rig_count.name, utilization_rate.name = 'Rig Count', 'Utilization Rate'
factor3 = lw.DualRelation(rig_count, utilization_rate)
fig = factor3.show()
fig.add_hline(
    secondary_y=True, 
    y=utilization_rate[utilization_rate.index >= datetime(clock.year - 5, clock.month, clock.day)].mean(),
    line_width=0.8,
    line_dash='dot'
)

### ☑️ 선물 곡선 (Backwardation)

*   **브렌트유 백워데이션 (Brent Backwardation)**(단위: $/bbl)<br>
브렌트유 근월물 가격이 원월물 가격보다 높은 상태. 미국 외 글로벌 원유 시장의 즉각적인 현물 공급 부족을 나타냄
*   **WTI유 백워데이션 (WTI Backwardation)**(단위: $/bbl)<br>
WTI유 근월물 가격이 원월물 가격보다 높은 상태. 미국 국내(특히 오클라호마 쿠싱 지역)의 강한 단기 현물 수요를 나타냄

---


| 구분 (행) | 백워데이션 증가 / 심화 (Increase) | 백워데이션 감소 / 축소 또는 콘탱고 전환 (Decrease) |
| :--- | :--- | :--- |
| **브렌트유 백워데이션**<br>(Brent Backwardation) | **📈 Long (매수) 신호**<br>· 글로벌 현물 원유의 극심한 공급 부족 시그널<br>· 유가 상승 압력 및 롤오버 효과 우호적 ➔ 에너지 섹터 상승 | **📉 Short (매도) 신호**<br>· 글로벌 공급 과잉 또는 단기 수요 둔화 시그널<br>· 유가 하락 압력 및 비축 수요 감소 ➔ 에너지 섹터 하락 |
| **WTI유 백워데이션**<br>(WTI Backwardation) | **📈 Long (매수) 신호**<br>· 미국 내 단기 원유 수요 급증 및 쿠싱 재고 고갈<br>· 미국 생산기업 가치 상승 유발 ➔ 에너지 섹터 상승 | **📉 Short (매도) 신호**<br>· 미국 내 공급 과잉 또는 정유사 가동 중단 시그널<br>· 선물 보유 비용(원월물 프리미엄) 발생 ➔ 에너지 섹터 하락 |


In [ ]:
month = 3
bz = oil_price['BZ=F'] - oil_price[future('BZ', month)]
cl = oil_price['CL=F'] - oil_price[future('CL', month)]
bz.name, cl.name = f'Brend-Spread({month}M)', f'WTI-Spread({month}M)'
factor = lw.DualRelation(bz, cl)
factor.show()

### ☑️ 달러와 생산 비용

*   **달러 인덱스 (DXY, US Dollar Index)**(단위: 포인트, pt)<br>
글로벌 주요 6개 통화에 대한 미국 달러의 가치를 지수화한 것. 원유를 포함한 국제 원자재 가격의 기준 통화 역할을 함.
*   **유·가스전 시추 생산자물가지수 (PPI, Drilling Oil and Gas Wells)**(단위: Index Dec 1985=100)<br>
석유 및 가스 시추 계약 서비스, 수평 시추 등 원유 개발 현장의 직접적인 생산 물가. 유전 서비스 및 탐사(E&P) 기업들의 비용 압박을 측정함.

---



| 구분 (행) | 지표 증가 / 상승 (Increase) | 지표 감소 / 하락 (Decrease) |
| :--- | :--- | :--- |
| **달러 인덱스**<br>(DXY) | **📉 Short (매도) 신호**<br>· 달러 강세로 인한 달러 표시 원유 가격의 상대적 하락 유발<br>· 글로벌 원유 구매력 약화 ➔ 에너지 섹터 하락 | **📈 Long (매수) 신호**<br>· 달러 약세로 원유를 포함한 위험자산 및 원자재 수요 유입<br>· 환율 효과로 인한 유가 상승 ➔ 에너지 섹터 상승 |
| **유·가스전 시추 PPI**<br>(PPI: Drilling Oil and Gas Wells) | **📉 Short (매도) 신호**<br>· 시추 비용(인건비, 장비대여료 등) 상승으로 E&P 기업 마진 압박<br>· 생산 원가 상승에 따른 투자 위축 ➔ 에너지 섹터 하락 | **📈 Long (매수) 신호**<br>· 시추 및 탐사 비용 안정화로 생산 효율성 및 이익률 개선<br>· 유전 서비스 및 탐사 기업의 마진 확대 ➔ 에너지 섹터 상승 |


In [ ]:
if isinstance(ppi, pd.DataFrame):
    ppi = ppi[ppi.columns[0]]
ppi.index = ppi.index.to_period('M').to_timestamp(freq='M')
dxy.name, ppi.name = 'DXY', 'PPI'
factor = lw.DualRelation(dxy, ppi)
factor.show()

## AUTO-PILOT

In [ ]:
import numpy as np
import pandas as pd


class EnergyMacroAnalyzer:

    def __init__(self, data: pd.DataFrame):
        """data: 10년치 시계열 데이터프레임 (날짜가 인덱스이거나 정렬되어 있어야 함)

        최근 5개 샘플을 추출하여 분석 진행
        """
        # 최근 5개 행만 추출 (가장 최신이 마지막 행에 위치한다고 가정)
        self.df = data.tail(5).copy()
        self.score_dict = {}
        self.details = {}

    def _check_trend(self, column_name: str) -> str:
        """최근 3~5개 샘플의 선형 추세(기울기)를 반환"""
        series = self.df[column_name].dropna()
        if len(series) < 3:
            return "Insufficient Data"
        x = np.arange(len(series))
        y = series.values
        slope = np.polyfit(x, y, 1)[0]
        return "UP" if slope > 0 else "DOWN"

    def analyze_commercial_inventory(self, col: str):
        """1. 상업용 원유 재고: 감소 추세가 우호적"""
        trend = self._check_trend(col)
        if trend == "DOWN":
            self.score_dict["Commercial_Inventory"] = 1.0  # 우호
            self.details["Commercial_Inventory"] = (
                "Bullish (재고 감소로 인한 공급 부족 압력)"
            )
        else:
            self.score_dict["Commercial_Inventory"] = 0.0  # 비우호
            self.details["Commercial_Inventory"] = (
                "Bearish (재고 증가로 인한 공급 과잉 우려)"
            )

    def analyze_spr_inventory(self, col: str):
        """2. SPR 원유 재고: 감소 추세는 일시적 공급 유입이나,

        총 재고량 감소 관점에서 감소를 유가 상방(우호)으로 해석하거나
        정부 비축 수요 관점으로 해석 가능. 여기서는 '재고 감소 = 우호' 기조 유지
        """
        trend = self._check_trend(col)
        if trend == "DOWN":
            self.score_dict["SPR_Inventory"] = 1.0
            self.details["SPR_Inventory"] = (
                "Bullish (정부 방출 지속 또는 총 재고 감소세)"
            )
        else:
            self.score_dict["SPR_Inventory"] = 0.0
            self.details["SPR_Inventory"] = "Bearish (정부 재비축 또는 공급 여력 증가)"

    def analyze_rig_count(self, col: str):
        """3. Baker Hughes Rig Count: 유가 상승기에도 증가 둔화/감소 시 공급 부족 지속으로 우호"""
        trend = self._check_trend(col)
        if trend == "DOWN":
            self.score_dict["Rig_Count"] = 1.0
            self.details["Rig_Count"] = (
                "Bullish (시추기 감소로 인한 미래 공급 통제)"
            )
        else:
            self.score_dict["Rig_Count"] = 0.0
            self.details["Rig_Count"] = "Bearish (시추기 증가로 인한 미래 공급 과잉 우려)"

    def analyze_futures_price(self, col: str):
        """4. WTI/Brent 선물 가격: 가격 상승세가 에너지 섹터 주가에 우호적"""
        trend = self._check_trend(col)
        if trend == "UP":
            self.score_dict["Futures_Price"] = 1.0
            self.details["Futures_Price"] = "Bullish (원유 선물 가격 단기 상승 추세)"
        else:
            self.score_dict["Futures_Price"] = 0.0
            self.details["Futures_Price"] = "Bearish (원유 선물 가격 단기 하락 추세)"

    def analyze_drilling_ppi(self, col: str):
        """5. Drilling PPI: 비용 지수이므로 하락해야 실질 마진(FCF)에 우호적"""
        trend = self._check_trend(col)
        if trend == "DOWN":
            self.score_dict["Drilling_PPI"] = 1.0
            self.details["Drilling_PPI"] = "Bullish (시추 비용 감소로 기업 FCF 개선)"
        else:
            self.score_dict["Drilling_PPI"] = 0.0
            self.details["Drilling_PPI"] = "Bearish (생산 비용 급등으로 기업 실질 마진 훼손)"

    def analyze_dxy(self, col: str):
        """6. DXY 달러 인덱스: 역상관관계이므로 달러 하락이 유가 상방 압력(우호)"""
        trend = self._check_trend(col)
        if trend == "DOWN":
            self.score_dict["DXY"] = 1.0
            self.details["DXY"] = (
                "Bullish (달러화 약세로 인한 원자재 가격 지지)"
            )
        else:
            self.score_dict["DXY"] = 0.0
            self.details["DXY"] = "Bearish (달러화 강세로 인한 원자재 가격 하락 압력)"

    def analyze_refinery_utilization(self, col: str):
        """7. 북미 정제시설 가동률: 90% 이상 유지 또는 상승 추세 시 실수요 강세로 우호"""
        latest_val = self.df[col].iloc[-1]
        trend = self._check_trend(col)

        if latest_val >= 90.0 or trend == "UP":
            self.score_dict["Refinery_Utilization"] = 1.0
            self.details["Refinery_Utilization"] = (
                f"Bullish (가동률 {latest_val}% 또는 상승세, 원유 소화력 양호)"
            )
        else:
            self.score_dict["Refinery_Utilization"] = 0.0
            self.details["Refinery_Utilization"] = (
                f"Bearish (가동률 {latest_val}% 및 하락세, 원유 수요 둔화 경고)"
            )

    def analyze_crack_spread(self, col: str):
        """8. Crack Spread (3-2-1): 정제마진 상승 추세가 정유사 수익성에 직결(우호)"""
        trend = self._check_trend(col)
        if trend == "UP":
            self.score_dict["Crack_Spread"] = 1.0
            self.details["Crack_Spread"] = "Bullish (정제마진 상승으로 정유사 수익성 개선)"
        else:
            self.score_dict["Crack_Spread"] = 0.0
            self.details["Crack_Spread"] = "Bearish (정제마진 하락, 에너지 주식 탈출 신호 가능성)"

    def run_full_analysis(self, col_mapping: dict):
        """모든 지표를 진단하고 종합 스코어를 백분율로 환산"""
        self.analyze_commercial_inventory(col_mapping["commercial_inventory"])
        self.analyze_spr_inventory(col_mapping["spr_inventory"])
        self.analyze_rig_count(col_mapping["rig_count"])
        self.analyze_futures_price(col_mapping["futures_price"])
        self.analyze_drilling_ppi(col_mapping["drilling_ppi"])
        self.analyze_dxy(col_mapping["dxy"])
        self.analyze_refinery_utilization(col_mapping["refinery_utilization"])
        self.analyze_crack_spread(col_mapping["crack_spread"])

        # 종합 점수 계산 (우호적 지표 개수 / 총 지표 개수 * 100)
        total_metrics = len(self.score_dict)
        bullish_count = sum(self.score_dict.values())
        final_score = (bullish_count / total_metrics) * 100

        print("=" * 65)
        print(f"★ 에너지 섹터 매크로 종합 점수: {final_score:.1f} / 100 점")
        if final_score >= 70:
            print("▶ 종합 의견: 대단히 우호적 (Strong Bullish) - 비중 확대 추천")
        elif final_score >= 40:
            print("▶ 종합 의견: 중립적 (Neutral) - 관망 또는 박스권 대응")
        else:
            print("▶ 종합 의견: 비우호적 (Bearish) - 리스크 관리 및 비중 축소")
        print("=" * 65)

        print("\n[지표별 세부 분석 현황]")
        for k, v in self.details.items():
            print(f"- {k:<22} : {v}")



    # # 사용자의 10년치 데이터프레임 중 최근 5개 데이터 상황을 모사한 Mock Data
    # mock_data = {
    #     "comm_inv": [450, 448, 445, 442, 440],  # 감소세 (우호)
    #     "spr_inv": [360, 360, 359, 358, 357],  # 감소세 (우호)
    #     "rig": [600, 598, 595, 596, 592],  # 감소세 (우호)
    #     "wti": [72.1, 73.5, 74.2, 76.0, 78.5],  # 상승세 (우호)
    #     "ppi": [120, 122, 125, 127, 130],  # 상승세 (비우호-비용증가)
    #     "dxy": [104.5, 104.2, 103.8, 103.5, 103.1],  # 하락세 (우호)
    #     "util": [88.5, 89.0, 89.5, 91.0, 92.5],  # 90% 돌파 및 상승 (우호)
    #     "crack": [24.5, 23.0, 22.1, 21.5, 20.2],  # 하락세 (비우호)
    # }
    # df_sample = pd.DataFrame(mock_data)

    # # 데이터프레임의 컬럼명 매핑 딕셔너리
    # column_mapping = {
    #     "commercial_inventory": "comm_inv",
    #     "spr_inventory": "spr_inv",
    #     "rig_count": "rig",
    #     "futures_price": "wti",
    #     "drilling_ppi": "ppi",
    #     "dxy": "dxy",
    #     "refinery_utilization": "util",
    #     "crack_spread": "crack",
    # }

    # # 분석기 실행
    # analyzer = EnergyMacroAnalyzer(df_sample)
    # analyzer.run_full_analysis(column_mapping)
